<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 19 · Credit Portfolios and Spread-Risk Management

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch19_credit_portfolios_spread_risk_management.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

This notebook scales credit analytics from single issuers to portfolios. You will

- compute CS01, the loss per basis point of spread widening, as market value times spread duration over 10,000,
- aggregate market value, CS01, and expected loss by rating bucket,
- apply rating-bucket spread stress scenarios and rank the resulting losses,
- compare candidate allocations on annual spread carry, an expected-loss proxy, and a credit-selloff loss for the same 10 million notional, and
- keep default risk (expected loss) separate from mark-to-market spread risk (CS01).

The portfolio and candidates are synthetic, so treat every number as a controlled teaching case.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The cell below runs the Chapter 19 script. It first aggregates market value and CS01 by rating for the synthetic portfolio, then computes portfolio losses under the three rating-bucket stress scenarios, and finally compares three candidate allocations on a common 10 million notional: annual spread carry, a rating-weighted expected-loss proxy, and a credit-selloff loss. Default probabilities are treated as one-year inputs throughout.

In [ ]:
SCRIPT = ROOT / "code" / "ch19_credit_portfolios_spread_risk_management.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

- Verify each issuer's CS01 as market value times spread duration over 10,000; the rating totals come to 4,650 for A, 3,820 for BBB, and 1,295 for BB.
- Check the stress ranking: mild widening 359,700, quality squeeze 683,500, credit selloff 894,100; the losses differ only through the per-bucket shock shapes.
- Compare the candidates on carry against risk: quality earns 155,000 in annual spread carry with a 639,600 selloff loss; income earns 270,000 with a 1,536,000 loss.
- Confirm the expected-loss proxy multiplies one-year PD by loss given default, and that spread carry is before default loss, funding, and costs.
- State the sign convention: positive CS01 times a positive spread widening gives a loss.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch19_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Portfolio Spread Duration and Loss Contribution

CS01 converts spread duration into a currency loss per basis point: market value times spread duration over 10,000. Under a rating-bucket selloff that widens A by 60 bp, BBB by 100 bp, and BB by 180 bp, each issuer's loss is its CS01 times its bucket's shock. The grouped view shows which rating drives the stress, even though the BB bucket holds the least market value in this illustrative portfolio.

In [ ]:
import pandas as pd

portfolio = pd.DataFrame({  # synthetic credit portfolio
    "issuer": ["Alpha", "Beta", "Gamma", "Delta"],
    "rating": ["A", "BBB", "BB", "A"],
    "market_value": [6.0e6, 5.0e6, 3.5e6, 4.5e6],
    "spread_duration": [4.3, 5.0, 3.7, 4.6],
})
portfolio["cs01"] = (  # loss per bp of widening, in currency
    portfolio.market_value * portfolio.spread_duration / 10_000
)
shock_bp = pd.Series({"A": 60, "BBB": 100, "BB": 180})  # selloff
portfolio["loss"] = portfolio.cs01 * portfolio.rating.map(shock_bp)
by_rating = portfolio.groupby("rating")[["cs01", "loss"]].sum()
print(by_rating.round(2))
print(f"total selloff loss: {portfolio.loss.sum():,.0f}")

### What to observe

CS01 is 2,580 for Alpha, 2,500 for Beta, 1,295 for Gamma, and 2,070 for Delta. Selloff losses are 154,800, 250,000, 233,100, and 124,200, totaling 762,100. The BB bucket contributes 233,100 from only 3.5 million of market value, because its 180 bp shock is three times the A bucket's 60 bp.

## Exercises

1. **CS01 by hand.** Compute CS01 for a 2 million position with spread duration 4.5. *Hint:* 2,000,000 times 4.5 over 10,000.
2. **Stress shape.** Re-rank the three scenarios by loss if the BB shock in the mild-widening scenario were doubled. *Hint:* reload `data/ch19_spread_stress_scenarios.csv` and adjust the BB column.
3. **Concentration check.** Aggregate the frozen portfolio in `data/ch19_credit_portfolio.csv` by sector and identify the largest sector by CS01.
4. **Candidate trade-off.** Divide each candidate's selloff loss by its annual spread carry and comment on what the ratio trades off. *Hint:* carry is quoted in currency per year, the loss is a one-off mark-to-market number.
5. **Migration exposure.** From `data/ch19_rating_transition_matrix.csv`, compute the one-year probability of a BBB issuer leaving the investment-grade bucket and explain the forced-sale channel.
6. **Two risks, one table.** In two sentences, explain how a portfolio can combine high expected loss with low CS01, or the reverse.

For more detail, see Chapter 19 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">